# Sample Training Data for Quality Labeling

Sample ~10K documents per 25-year period from the **English historical corpus**.
Only samples from clean documents (uses cleaning masks from `Clean_Data.ipynb`).
Samples are **equally distributed across years** within each period for balanced temporal coverage.

**Prerequisite:** Run `Clean_Data.ipynb` first to generate cleaning masks.

**How it works:**

1. **Count** clean docs per year by reading mask file metadata (no I/O, just row counts)
2. **Allocate** 10K / num_active_years = fixed samples per year (capped by available clean docs)
3. **Sample** per year: load mask -> get sorted clean row indices -> pick random indices -> read only those rows from raw parquet
4. **Shuffle** and save

**Source:** `D:\hist_LLM\corpus\raw\{year}\subset_*.parquet` (columns: `text`, `identifier`)
**Masks:** `D:\hist_LLM\corpus\cleaning_masks\{year}\{subset}_mask.parquet`
**Output:** `D:\hist_LLM\processing\sample_data\training_samples_{period}.parquet` -- schema: `[text, original_index, year]`

In [1]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import numpy as np
import gc
from tqdm.auto import tqdm

# --- CONFIG ---
RAW_DIR = Path(r"D:\hist_LLM\corpus\raw")
MASKS_DIR = Path(r"D:\hist_LLM\corpus\cleaning_masks")
OUTPUT_DIR = Path(r"D:\hist_LLM\processing\sample_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = 10_000
RANDOM_STATE = 42


def count_clean(year: int) -> int:
    """Count clean docs for a year by reading mask file metadata (no actual I/O)."""
    mask_dir = MASKS_DIR / str(year)
    if not mask_dir.exists():
        return 0
    total = 0
    for f in mask_dir.glob("*_mask.parquet"):
        try:
            total += pq.ParquetFile(f).metadata.num_rows
        except:
            pass
    return total


def sample_year(year: int, n: int, rng: np.random.Generator) -> list:
    """Sample n clean docs from raw parquets for a given year.

    1. Load clean row indices from masks for each subset file
    2. Build a flat list of (file, sorted_clean_indices) pairs
    3. Pick n random positions from the global clean index space
    4. Map global positions back to per-file row indices
    5. Read only the selected rows from the raw parquet files
    """
    year_dir = RAW_DIR / str(year)
    mask_dir = MASKS_DIR / str(year)
    if not year_dir.exists() or not mask_dir.exists():
        return []

    files = sorted(year_dir.glob("subset_*.parquet"))
    if not files:
        return []

    # Build (file, sorted_clean_indices) pairs from masks
    file_clean_indices = []
    for f in files:
        mask_path = mask_dir / f"{f.stem}_mask.parquet"
        if mask_path.exists():
            mask_df = pd.read_parquet(mask_path)
            clean_idx = sorted(mask_df["original_index"].tolist())
            if clean_idx:
                file_clean_indices.append((f, clean_idx))

    total_clean = sum(len(idx) for _, idx in file_clean_indices)
    if total_clean == 0:
        return []

    n = min(n, total_clean)

    # Pick n random positions from global clean index space
    chosen = set(rng.choice(total_clean, size=n, replace=False))

    # Map global positions to per-file row indices
    samples = []
    offset = 0
    for f, clean_idx in file_clean_indices:
        file_chosen = [clean_idx[i - offset] for i in chosen if offset <= i < offset + len(clean_idx)]
        offset += len(clean_idx)
        if not file_chosen:
            continue

        df = pd.read_parquet(f, columns=["identifier", "text"])
        selected = df.loc[df.index.isin(file_chosen)]
        for _, row in selected.iterrows():
            text = row["text"]
            if text and str(text).strip():
                samples.append({
                    "text": text,
                    "original_index": row["identifier"],
                    "year": year,
                })
        del df, selected

    gc.collect()
    return samples


def sample_period(target_years: list):
    """Sample 10K clean docs, equally distributed across years in the period."""
    rng = np.random.default_rng(RANDOM_STATE)

    # Phase 1: Count clean docs per year (to know upper bound)
    print("Counting clean documents...", flush=True)
    year_counts = {}
    for year in tqdm(target_years, desc="Counting"):
        year_counts[year] = count_clean(year)

    # Only sample from years that have clean docs
    active_years = [y for y in target_years if year_counts[y] > 0]
    total_clean = sum(year_counts[y] for y in active_years)

    if total_clean == 0:
        print("No clean documents found! Run Clean_Data.ipynb first.")
        return

    print(f"Total clean docs: {total_clean:,} across {len(active_years)} years")

    # Phase 2: Equal allocation per year, capped by available clean docs
    per_year = SAMPLE_SIZE // len(active_years)
    remainder = SAMPLE_SIZE % len(active_years)

    print(f"Target: {per_year} samples/year ({len(active_years)} years)")
    print("Sampling...", flush=True)

    all_samples = []
    for i, year in enumerate(tqdm(active_years, desc="Sampling")):
        # Distribute remainder across first few years
        year_n = per_year + (1 if i < remainder else 0)
        year_n = min(year_n, year_counts[year])
        samples = sample_year(year, year_n, rng)
        all_samples.extend(samples)

    # Phase 3: Shuffle and save
    period_name = f"{min(target_years)}_{max(target_years)}"
    output_path = OUTPUT_DIR / f"training_samples_{period_name}.parquet"

    final_df = pd.DataFrame(all_samples)
    final_df = final_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    final_df.to_parquet(output_path, index=False)

    print(f"Saved {len(final_df)} samples to {output_path.name}")

In [2]:
# 14 periods: 1678-1700, then 25-year blocks through 2023
all_periods = [range(1678, 1701)]
for start in range(1701, 2002, 25):
    end = min(start + 24, 2023)
    all_periods.append(range(start, end + 1))

for period_range in all_periods:
    print(f"\n{'='*60}")
    print(f"Period: {min(period_range)}_{max(period_range)}")
    print(f"{'='*60}")
    sample_period(list(period_range))


Period: 1678_1700
Counting clean documents...


Counting:   0%|          | 0/23 [00:00<?, ?it/s]

Total clean docs: 67,444 across 23 years
Target: 434 samples/year (23 years)
Sampling...


Sampling:   0%|          | 0/23 [00:00<?, ?it/s]


KeyboardInterrupt



In [3]:
 sample_period(list(range(2001, 2024)))

Counting clean documents...


Counting:   0%|          | 0/23 [00:00<?, ?it/s]

Total clean docs: 35,754,392 across 23 years
Target: 434 samples/year (23 years)
Sampling...


Sampling:   0%|          | 0/23 [00:00<?, ?it/s]

Saved 10000 samples to training_samples_2001_2023.parquet
